In [2]:
!pip install altair

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/731.2 kB ? eta -:--:--
   --------------------------------------- 731.2/731.2 kB 10.0 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\Amirhosein Rostami\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import altair as alt

In [2]:
xl_file = pd.ExcelFile("./OntarioLead3_5_25.xlsx")

dfs = {
    sheet_name: xl_file.parse(sheet_name) for sheet_name in xl_file.sheet_names
}

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [4]:
master = dfs["Master"]

In [5]:
master = master[master.columns.drop(list(master.filter(regex='Unnamed*')))]

In [6]:
drop = []
for column in master.columns:
    if column not in ['Year', 'DWS Name', 'Owner Legal Name', 'DWS Category', 'Sample Date', 'Sample Type Name', 'Result', 'Exceed2']:
        drop.append(column)
master = master[master.columns.drop(drop)]

In [7]:
def extract(dws_name):
    proned_name = " ".join(dws_name.split(" ")[1:])
    return proned_name.strip()

master.insert(loc=4, column="DWS Name cleared", value=master["DWS Name"].apply(lambda x: extract(x)))
master = master[master.columns.drop("DWS Name")]

In [8]:
master.loc[:, 'Sample Date'] = pd.to_datetime(master['Sample Date'])
master['Sample Date'].dtype

dtype('O')

In [9]:
master.head()

,Year,Owner Legal Name,DWS Category,DWS Name cleared,Sample Date,Sample Type Name,Result,Exceed2
0,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S (5473),2020-03-15 11:02:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N
1,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S (5473),2020-03-15 10:22:00,PLUMBING - STANDING DRINKING WATER,0.5,N
2,201920,Lambton-Kent District School Board,Public School,ALEXANDER MACKENZIE SS (5204),2020-03-15 09:19:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N
3,201920,Lambton-Kent District School Board,Public School,ALEXANDER MACKENZIE SS (5204),2020-03-15 09:18:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N
4,201920,Lambton-Kent District School Board,Public School,ALEXANDER MACKENZIE SS (5204),2020-03-15 08:39:00,PLUMBING - STANDING DRINKING WATER,0.8,N


In [10]:
for index, row in master.iterrows():
    if row['Result'] == 5:
        # print("School Name: ", row['Owner Legal Name'])
        master.at[index, 'Exceed2'] = 'N'  # Change value in column Y

In [11]:
master[master['DWS Name cleared'] == "BAYSHORE PS (143)"].tail()

,Year,Owner Legal Name,DWS Category,DWS Name cleared,Sample Date,Sample Type Name,Result,Exceed2
89467,202122,OTTAWA CARLETON DISTRICT SCHOOL BOARD,Public School,BAYSHORE PS (143),2021-08-05 07:46:00,PLUMBING - STANDING DRINKING WATER,7.91,Y
117911,202223,OTTAWA CARLETON DISTRICT SCHOOL BOARD,Public School,BAYSHORE PS (143),2022-06-01 06:36:00,PLUMBING - STANDING DRINKING WATER,5.00,N
117912,202223,OTTAWA CARLETON DISTRICT SCHOOL BOARD,Public School,BAYSHORE PS (143),2022-06-01 07:39:00,PLUMBING - FLUSHED DRINKING WATER,1.00,N
152712,202324,OTTAWA CARLETON DISTRICT SCHOOL BOARD,Public School,BAYSHORE PS (143),1970-01-01 00:00:00.000045086,PLUMBING - STANDING DRINKING WATER,1.00,N
152713,202324,OTTAWA CARLETON DISTRICT SCHOOL BOARD,Public School,BAYSHORE PS (143),1970-01-01 00:00:00.000045086,PLUMBING - FLUSHED DRINKING WATER,1.00,N


In [12]:
master.to_excel("new_year_refined_master.xlsx", sheet_name="master")

In [13]:
xl_file = pd.ExcelFile("./new_year_refined_master.xlsx")

dfs = {
    sheet_name: xl_file.parse(sheet_name) for sheet_name in xl_file.sheet_names
}
master = dfs["master"]
master = master[master.columns.drop(list(master.filter(regex='Unnamed*')))]
master.head()

,Year,Owner Legal Name,DWS Category,DWS Name cleared,Sample Date,Sample Type Name,Result,Exceed2
0,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S (5473),2020-03-15 11:02:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N
1,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S (5473),2020-03-15 10:22:00,PLUMBING - STANDING DRINKING WATER,0.5,N
2,201920,Lambton-Kent District School Board,Public School,ALEXANDER MACKENZIE SS (5204),2020-03-15 09:19:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N
3,201920,Lambton-Kent District School Board,Public School,ALEXANDER MACKENZIE SS (5204),2020-03-15 09:18:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N
4,201920,Lambton-Kent District School Board,Public School,ALEXANDER MACKENZIE SS (5204),2020-03-15 08:39:00,PLUMBING - STANDING DRINKING WATER,0.8,N


In [14]:
years = master["Year"].unique().tolist()
categories = master["DWS Category"].unique().tolist()
dws_name = master["DWS Name cleared"].unique().tolist()
print("Year: ", years)
print("Categories: ", categories)
print("Len names: ", len(dws_name))

Year:  [201920, 202021, 202122, 202223, 202324]
Categories:  ['Public School', 'Child care Centre', 'Private School']
Len names:  8907


In [15]:
def get_result(school_name):
    answer = {}
    if school_name == "ALL":
        result = master[master["Year"].isin(years)][master["DWS Category"].isin(categories)]
    else:
        result = master[master["Year"].isin(years)][master["DWS Category"].isin(categories)][master["DWS Name cleared"] == school_name]

    if not school_name == "ALL":
        owner_legal_name = result['Owner Legal Name'].iloc[0]
        dws_category = result['DWS Category'].iloc[0]
        school_name = school_name

    answer["section0"] = {
        "School Name": school_name if not school_name == "ALL" else "ALL",
        "Owner Legal Name": owner_legal_name if not school_name == "ALL" else "ALL",
        "School Category": dws_category if not school_name == "ALL" else "ALL",
    }

    pie_data = pd.DataFrame(result.groupby(['Exceed2'])['Exceed2'].count())
    pie_result = {
        "N": 0,
        "Y": 0,
    }
    for i in range(len(pie_data)):
        pie_result[pie_data.iloc[i].name] += int(pie_data.iloc[i]["Exceed2"])

    pie_result["total tests"] = pie_result["N"] + pie_result["Y"]
    pie_result["exceedance"] = pie_result["Y"]
    pie_result["failure rate"] = 0 if (pie_result["Y"]+pie_result["N"]) == 0 else (pie_result["Y"]/(pie_result["Y"]+pie_result["N"]))*100
    pie_result["failure rate"] = round(pie_result["failure rate"],2)
    answer["section1"] = pie_result

    year_exceed = pd.DataFrame(result.groupby(['Year','Exceed2'])['Exceed2'].count())
    years_ratio = {
        year: {
            "N": 0,
            "Y": 0
        } for year in years
    }
    for i in range(len(year_exceed)):
        years_ratio[year_exceed.iloc[i].name[0]][year_exceed.iloc[i].name[1]] += year_exceed.iloc[i][0]

    for year, distribution in years_ratio.items():
        years_ratio[year] = {
            "total tests": int(distribution["Y"] + distribution["N"]),
            "exceedance": int(distribution["Y"]),
            "failure rate": 0 if (distribution["Y"] + distribution["N"]) == 0 else round((distribution["Y"]/(distribution["Y"] + distribution["N"]))*100,2)
        
        }
    answer["section2"] = years_ratio
    if school_name == "ALL":
        answer["section3"] = []
        return answer

    result = result.sort_values(by=['Year', 'Sample Date'], ascending=True)
    records = result[['Sample Date', 'Result', 'Sample Type Name']]
    import json
    json_records = []
    for i in range(len(records)): 
        d = json.loads(records.iloc[i, :].to_json())
        json_records.append(d)
    # if len(json_records) > 5:
    #     json_records = json_records[0:5]
    answer["section3"] = json_records
    
    return answer

In [16]:
from tqdm import tqdm
final_school_dump = {}
for dws in tqdm(dws_name):
    if not str(dws) == "nan":
        final_school_dump[dws] = get_result(dws)

100%|██████████| 8907/8907 [07:25<00:00, 19.98it/s]


In [17]:
# aggregated ALL 
final_school_dump["ALL"] = get_result("ALL")

In [18]:
final_school_dump["ALL"]

{'section0': {'School Name': 'ALL',
  'Owner Legal Name': 'ALL',
  'School Category': 'ALL'},
 'section1': {'N': 143346,
  'Y': 15362,
  'total tests': 158708,
  'exceedance': 15362,
  'failure rate': 9.68},
 'section2': {201920: {'total tests': 50751,
   'exceedance': 4829,
   'failure rate': 9.52},
  202021: {'total tests': 27527, 'exceedance': 3725, 'failure rate': 13.53},
  202122: {'total tests': 31712, 'exceedance': 3796, 'failure rate': 11.97},
  202223: {'total tests': 27155, 'exceedance': 1686, 'failure rate': 6.21},
  202324: {'total tests': 21563, 'exceedance': 1326, 'failure rate': 6.15}},
 'section3': []}

In [19]:
import json
with open("new_year_final_last.json", "w") as outfile:
    json.dump(final_school_dump, outfile)

In [18]:
# multi select
_years = years # [201920]
_categories = categories # ['Public School']
_school_name = 'LAMBTON KENT COMP S'

In [57]:
result = master[master["Year"].isin(_years)][master["DWS Category"].isin(_categories)][master["DWS Name cleared"] == _school_name]

In [21]:
result.head()

,Year,Owner Legal Name,DWS Category,DWS Name cleared,Sample Date,Sample Type Name,Result,Exceed2
0,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S,2020-03-15 11:02:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N
1,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S,2020-03-15 10:22:00,PLUMBING - STANDING DRINKING WATER,0.5,N
32,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S,2020-03-08 13:32:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N
33,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S,2020-03-08 12:52:00,PLUMBING - STANDING DRINKING WATER,0.5,N
93,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S,2020-03-01 09:34:00,PLUMBING - FLUSHED DRINKING WATER,0.5,N


In [29]:
pie_data = pd.DataFrame(result.groupby(['Exceed2'])['Exceed2'].count())
pie_result = {
    "N": 0,
    "Y": 0,
}
for i in range(len(pie_data)):
    pie_result[pie_data.iloc[i].name] = pie_data.iloc[i]["Exceed2"]

pie_result["total tests"] = pie_result["N"] + pie_result["Y"]
pie_result["exceedance"] = pie_result["Y"]
pie_result["failure rate"] = (pie_result["Y"]/(pie_result["Y"]+pie_result["N"]))*100
pie_result["failure rate"] = round(pie_result["failure rate"],2)
pie_result

{'N': 29, 'Y': 5, 'total tests': 34, 'exceedance': 5, 'failure rate': 14.71}

In [31]:
year_exceed = pd.DataFrame(result.groupby(['Year','Exceed2'])['Exceed2'].count())
years_ratio = {
    year: {
        "N": 0,
        "Y": 0
    } for year in years
}
for i in range(len(year_exceed)):
    years_ratio[year_exceed.iloc[i].name[0]][year_exceed.iloc[i].name[1]] += year_exceed.iloc[i][0]

criteria = "N"
for year, distribution in years_ratio.items():
    years_ratio[year] = {
        "total tests": distribution["Y"] + distribution["N"],
        "exceedance": distribution["Y"],
        "failure rate": round((distribution["Y"]/(distribution["Y"] + distribution["N"]))*100,2)
    
    }
years_ratio

{201920: {'total tests': 14, 'exceedance': 0, 'failure rate': 0.0},
 202021: {'total tests': 14, 'exceedance': 2, 'failure rate': 14.29},
 202122: {'total tests': 4, 'exceedance': 2, 'failure rate': 50.0},
 202223: {'total tests': 2, 'exceedance': 1, 'failure rate': 50.0}}

In [63]:
result = result.sort_values(by=['Year', 'Sample Date'], ascending=True)
result.head(n=2)

,Year,Owner Legal Name,DWS Category,DWS Name cleared,Sample Date,Sample Type Name,Result,Exceed2
16674,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S,2019-06-16 12:18:00,PLUMBING - STANDING DRINKING WATER,3.01,N
16687,201920,Lambton-Kent District School Board,Public School,LAMBTON KENT COMP S,2019-06-16 12:58:00,PLUMBING - FLUSHED DRINKING WATER,1.35,N


In [59]:
result = result.sort_values(by=['Year', 'Sample Date'], ascending=True)
records = result[['Sample Date', 'Result', 'Sample Type Name']]


In [64]:
import json
json_records = []
for i in range(len(records)): 
    d = json.loads(records.iloc[i, :].to_json())
    json_records.append(d)

In [11]:
pie_data = pd.DataFrame(pie_result.items(), columns=['Label', 'Count'])

pie = alt.Chart(pie_data).mark_arc(innerRadius=75).encode(
    theta=alt.Theta(field="Count", type="quantitative"),
    color=alt.Color(title="Label", field="Label", type="nominal", scale=alt.Scale(scheme='purples')),
    tooltip = ['Count']
    ).properties(
    width=600,
    height=400,
    title='Label Distribution'
    ).configure_title(
        fontSize=20,
        font='Helvetica',
        fontWeight='bold',
        color='black'
    )
pie.save("pie.html", format="html")

c:\Users\AmirHossein\AppData\Local\Programs\Python\Python310\lib\site-packages\altair\utils\core.py:317: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for col_name, dtype in df.dtypes.iteritems():


In [12]:
colors = ['#FF5733', '#C70039', '#900C3F', '#581845', '#36404D']
histogram = alt.Chart(result).mark_bar(
    color=colors[0],
    opacity=0.7
).encode(
    alt.X('Result', bin=alt.Bin(step=1), title='Lead (UG/L)'),
    alt.Y('count()', title='Count')
).properties(
    width=600,
    height=400,
    title='Lead Distribution'
).configure_title(
    fontSize=20,
    font='Helvetica',
    fontWeight='bold',
    color=colors[1]
).configure_axis(
    labelFontSize=12,
    titleFontSize=16,
    labelColor=colors[2],
    titleColor=colors[3]
)
histogram.save("hist.html", format="html")

In [13]:
year_exceed = pd.DataFrame(result.groupby(['Year','Exceed2'])['Exceed2'].count())
years_ratio = {
    year: {
        "N": 0,
        "Y": 0
    } for year in years
}
for i in range(len(year_exceed)):
    years_ratio[year_exceed.iloc[i].name[0]][year_exceed.iloc[i].name[1]] += year_exceed.iloc[i][0]

criteria = "N"
for year, distribution in years_ratio.items():
    years_ratio[year] = distribution[criteria]/(distribution[criteria] + distribution["Y" if criteria == "N" else "N"])

# Sample data for a line chart
line_data = pd.DataFrame({
    'Year': [str(year) for year in years_ratio.keys()],
    f'{criteria} ratio': years_ratio.values()
})

# Create a line chart using Altair
line_chart = alt.Chart(line_data).mark_line(
    color=colors[0]
).encode(
    x='Year',
    y=f'{criteria} ratio'
).properties(
    width=600,
    height=400,
    title=f'{criteria} ratio over the years'
).configure_title(
    fontSize=20,
    font='Helvetica',
    fontWeight='bold',
    color=colors[1]
).configure_axis(
    labelFontSize=12,
    titleFontSize=16,
    labelColor=colors[2],
    titleColor=colors[3]
)
line_chart.save("line.html", format="html")

c:\Users\AmirHossein\AppData\Local\Programs\Python\Python310\lib\site-packages\altair\utils\core.py:317: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for col_name, dtype in df.dtypes.iteritems():
